In [ ]:
import importlib

import lightning.pytorch as pl
import mlflow.pyfunc
import pandas as pd
import scipy.stats
import torch
import torch_geometric as pyg

from sportsml.graph.datamodule import GraphDataModule
from sportsml.graph.graph import create_complete_graph
from sportsml.graph.model import GraphModel, SportsMLPredictor
from sportsml.graph.nn.encoder.edge_encoder import EdgeEncoder
from sportsml.graph.nn.encoder.mean import EdgeMean
from sportsml.graph.nn.predictor.ffn import EdgeFFN


In [ ]:
SPORT = "cbb"

features_module = importlib.import_module(f"sportsml.{SPORT}.data.features")

In [ ]:
games = pd.read_csv(f"../data/{SPORT}/raw.csv").dropna(
    subset=features_module.GRAPH_STATS_COLUMNS + [features_module.TARGET_COLUMN]
)

dm = GraphDataModule(
    games=games,
    stats_columns=features_module.GRAPH_STATS_COLUMNS,
    target_column=features_module.TARGET_COLUMN,
    season_column=features_module.SEASON_COLUMN,
    date_column=features_module.DATE_COLUMN,
    batch_size=4,
)

In [ ]:
model = GraphModel(
    encoder=EdgeMean(in_edge_channels=len(features_module.GRAPH_STATS_COLUMNS)),
    predictor=EdgeFFN(
        in_dim=len(features_module.GRAPH_STATS_COLUMNS), hidden_dim=100, out_dim=1
    ),
)

In [ ]:
trainer = pl.Trainer(
    max_epochs=100,
    callbacks=[
        pl.callbacks.EarlyStopping(monitor="val_loss", patience=5),
        pl.callbacks.ModelCheckpoint(dirpath="models", monitor="val_loss"),
    ],
)

In [ ]:
trainer.fit(model, dm)

In [ ]:
trainer.test(model, dm, ckpt_path='best', weights_only=False)

In [ ]:
model = GraphModel.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)

In [ ]:
ge = dm.get_latest_graph()
x = model.encoder(edge_attr=ge.edge_attr, edge_index=ge.edge_index)

predictor = SportsMLPredictor(model=model, team_embeddings=x)

In [ ]:
gp = create_complete_graph(ge.num_nodes)
model_input = pd.DataFrame({
    'team': gp.edge_index[1],
    'opp': gp.edge_index[0],
})

In [ ]:
preds = predictor.predict(model_input)

In [ ]:
mlflow.pyfunc.save_model('mlflow/models/cbb', python_model=predictor)

In [ ]:
predictor = mlflow.pyfunc.load_model('..//models/cbb/pyg')

In [ ]:
preds = predictor.predict(model_input)
preds['win_prob'] = scipy.stats.norm.cdf(preds['preds'] / preds['preds'].std())

In [ ]:
preds = preds.pivot_table(values='win_prob', index='team', columns='opp')

In [ ]:
from sportsml.cbb.data.nodes import team_name_map

In [ ]:
preds = preds.rename(index=team_name_map, columns=team_name_map)

In [ ]:
preds = preds.loc[preds.mean(axis=0).sort_values().index, preds.mean(axis=0).sort_values().index]

In [ ]:
preds